# Capital Markets Agent - Interactive Testing

This notebook provides an interactive interface to test the deployed Bedrock AgentCore runtime for financial analysis of S&P 100 companies.

## Prerequisites

1. Complete all setup steps from the README
2. CDK stack deployed successfully
3. TigerGraph populated with data
4. Knowledge Base sync completed
5. AWS credentials configured

## Setup

In [ ]:
# Install required packages (run once)
!pip install boto3 -q

## AWS Credentials Setup

Configure AWS credentials **before** launching Jupyter using one of these methods:

**Option 1: AWS CLI (Recommended)**
```bash
aws configure
```

**Option 2: AWS SSO**
```bash
aws sso login --profile your-profile
export AWS_PROFILE=your-profile
```

**Option 3: Environment variables** (set in terminal before launching Jupyter)
```bash
export AWS_ACCESS_KEY_ID=your-key
export AWS_SECRET_ACCESS_KEY=your-secret
```

⚠️ **Never hardcode credentials in notebooks or commit them to version control.**

In [ ]:
import boto3

# Verify AWS credentials are configured
# boto3 automatically uses credentials from AWS CLI, SSO, env vars, or IAM role
try:
    sts = boto3.client('sts', region_name='us-east-1')
    identity = sts.get_caller_identity()
    print(f"✅ Authenticated as: {identity['Arn']}")
except Exception as e:
    print(f"❌ Not authenticated. Configure credentials first:")
    print("   Run: aws configure")
    print("   Or:  aws sso login")
    raise

In [ ]:
import json
import uuid
from datetime import datetime
from IPython.display import display, Markdown

# Initialize Bedrock AgentCore client
client = boto3.client('bedrock-agentcore', region_name='us-east-1')

print("✅ Bedrock AgentCore client initialized")

## Configuration

Get these values from your CDK deployment outputs or AWS Console:
- Agent Runtime ARN: CloudFormation → Outputs → Look for runtime ARN
- Session ID: Can be any string 33+ characters (generated below)
- Endpoint: Optional, defaults to "DEFAULT" if not specified

In [ ]:
# CONFIGURATION - Update with your values
AGENT_RUNTIME_ARN = 'arn:aws:bedrock-agentcore:us-east-1:YOUR_ACCOUNT_ID:runtime/tenkAnalyzerAgent_Agent-XXXXX'

# Generate a unique session ID (33+ characters)
# Each new session ID creates a new MicroVM - reuse for conversation continuity
SESSION_ID = f"test-session-{uuid.uuid4()}"

# Endpoint qualifier (optional)
ENDPOINT = "DEFAULT"  # Use "DEFAULT" or specify a custom endpoint

print(f"Session ID: {SESSION_ID}")
print(f"Session ID Length: {len(SESSION_ID)} characters")

## Helper Function

In [ ]:
def query_agent(prompt, verbose=True):
    """
    Query the Capital Markets Agent with a financial analysis question.
    
    Args:
        prompt (str): Your question about S&P 100 companies
        verbose (bool): Print detailed response info
    
    Returns:
        dict: Agent response
    """
    if verbose:
        print(f"\n{'='*80}")
        print(f"QUERY: {prompt}")
        print(f"{'='*80}\n")
        print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Session: {SESSION_ID[:30]}...\n")
    
    try:
        # Prepare payload
        payload = json.dumps({"prompt": prompt})
        
        # Invoke agent
        response = client.invoke_agent_runtime(
            agentRuntimeArn=AGENT_RUNTIME_ARN,
            runtimeSessionId=SESSION_ID,
            payload=payload,
            qualifier=ENDPOINT
        )
        
        # Parse SSE (Server-Sent Events) streaming response
        response_body = b""
        for event in response['response']:
            response_body += event
        
        # Parse SSE format: extract text from 'data: "..."' lines
        text_parts = []
        for line in response_body.decode('utf-8').split('\n'):
            if line.startswith('data: '):
                try:
                    # Extract JSON string after 'data: '
                    text_parts.append(json.loads(line[6:]))
                except json.JSONDecodeError:
                    pass
        
        # Combine all text parts
        full_response = ''.join(text_parts)
        response_data = {"response": full_response}
        
        if verbose:
            print("\n" + "="*80)
            print("AGENT RESPONSE:")
            print("="*80 + "\n")
            
            # Extract and render response as markdown
            response_text = response_data.get('response', response_data.get('output', str(response_data)))
            display(Markdown(response_text))
            
            print("\n" + "="*80)
        
        return response_data
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        print("\nTroubleshooting:")
        print("1. Verify AGENT_RUNTIME_ARN is correct")
        print("2. Check AWS credentials are configured")
        print("3. Ensure agent runtime is READY (check AWS Console)")
        print("4. Verify TigerGraph is running on EC2")
        print("5. Confirm Knowledge Base sync completed\n")
        return None

## Quick Test

Verify the agent is working with a simple query:

In [ ]:
# Simple connectivity test
response = query_agent("What are the main risk factors for Apple?")

## Example Queries

### Company-Specific Analysis

In [ ]:
# Apple risk analysis
query_agent("What are the main risk factors for Apple?")

In [ ]:
# Microsoft ESG initiatives
query_agent("What ESG initiatives does Microsoft mention in their filings?")

In [ ]:
# Amazon supply chain
query_agent("Describe Amazon's supply chain dependencies and risks")

### Sector & Industry Analysis

In [ ]:
# Geographic exposure
query_agent("Which S&P 100 companies operate in China?")

In [ ]:
# Supply chain comparison
query_agent("Compare supply chain dependencies between tech companies")

In [ ]:
# Semiconductor exposure
query_agent("Find companies exposed to semiconductor shortages")

### Regulatory & Compliance

In [ ]:
# Regulatory issues
query_agent("What are the main regulatory issues for the S&P 100 companies?")

In [ ]:
# Financial crime exposure
query_agent("Which companies are most exposed to financial crime?")

In [ ]:
# Financial institution risks
query_agent("What are the top risk factors for financial institutions?")

### Strategic & Market Analysis

In [ ]:
# AI/Technology adoption
query_agent("Which companies are investing most heavily in AI and machine learning?")

In [ ]:
# Climate risk
query_agent("What climate-related risks do energy companies face?")

In [ ]:
# Cybersecurity risks
query_agent("Which companies mention cybersecurity as a major risk factor?")

## Custom Query

Enter your own question:

In [ ]:
# Your custom query
custom_query = "Your question here"
query_agent(custom_query)

## Multi-Turn Conversation

The agent maintains memory across queries in the same session. Try a follow-up question:

In [ ]:
# Initial query
query_agent("What are Tesla's main risk factors?")

In [ ]:
# Follow-up (agent remembers previous context)
query_agent("How do these compare to other automotive companies?")

## Batch Queries

Run multiple queries programmatically:

In [ ]:
# Batch analysis
queries = [
    "What are the main risks for banks?",
    "Which tech companies face antitrust concerns?",
    "What are common supply chain risks across industries?"
]

results = {}
for i, q in enumerate(queries, 1):
    print(f"\n{'#'*80}")
    print(f"Query {i}/{len(queries)}")
    print(f"{'#'*80}")
    results[q] = query_agent(q)

print(f"\n\nCompleted {len(results)} queries")

## Session Management

Create a new session to start fresh (clears conversation memory):

In [ ]:
# Generate new session ID for fresh start
SESSION_ID = f"test-session-{uuid.uuid4()}"
print(f"New Session ID: {SESSION_ID}")
print("Memory cleared - starting fresh conversation")

## Tips & Best Practices

### Query Formulation
- Be specific about what companies or sectors you're analyzing
- Ask for comparisons across companies when relevant
- Request specific evidence or citations for claims
- Use follow-up questions to dive deeper into specific topics

### Agent Capabilities
- **Graph Analysis**: Entity relationships, risk patterns, industry trends
- **Document Retrieval**: Exact quotes from 10-K filings, detailed disclosures
- **Synthesis**: Combines structured and unstructured data for comprehensive analysis
- **Memory**: Maintains conversation context across queries in the same session

### Troubleshooting
If queries fail:
1. Check AGENT_RUNTIME_ARN is correct
2. Verify AWS credentials: `aws sts get-caller-identity`
3. Confirm agent status in AWS Console (should be READY)
4. Check TigerGraph is running: `aws ssm start-session --target <TigerGraphInstanceId>`
5. Verify Knowledge Base sync completed successfully